In [ ]:
!pip install groq python-dotenv numpy tqdm datasets

In [ ]:
from groq import Groq
from dotenv import load_dotenv
from datasets import load_dataset

import os
from tqdm import tqdm
import re
import random
import pprint

from typing import List, Dict, Any

load_dotenv()
random.seed(0)

client = Groq()
gsm8k_dataset = load_dataset("gsm8k", "main")

gsm8k_train = gsm8k_dataset["train"]
gsm8k_test  = gsm8k_dataset["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def generate_response_using_Llama(
        prompt: str,
        model: str = "llama-3.1-8b-instant"
    ):
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful assistant that solves math problems."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            model=model,
            temperature=0.3, ### 수정해도 됩니다!
            stream=False
        )
        return chat_completion.choices[0].message.content

    except Exception as e:
        print(f"API call error: {str(e)}")
        return None

#### 응답 잘 나오는지 확인해보기

In [ ]:
response = generate_response_using_Llama(
    prompt="Hello world!",
)
print(response)

Hello world! It's nice to meet you. I'm here to help with any math problems you might have. What's on your mind? Do you have a specific problem you'd like help with, or would you like me to suggest some examples?


#### GSM8K 데이터셋 확인해보기

In [ ]:
print("[Question]")
for l in gsm8k_test['question'][0].split("."):
    print(l)
print("="*100)
print("[Answer]")
print(gsm8k_test['answer'][0])

[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


#### Util 함수들
- extract_final_answer: LLM의 응답을 parse하여 최종 결과만 추출 (정답과 비교하기 위해)
- run_benchmark_test: 벤치마크 테스트
- save_final_result: 결과물 제출을 위한 함수

In [ ]:
### 수정해도 됩니다!
def extract_final_answer(response: str):
    if not response:
        return None

    # 정답 태그 뒤의 숫자나 마지막에 등장하는 숫자를 찾음
    regex = r"(?:Answer:|Model response:)\s*\$?([0-9,.]+)\b"
    matches = re.findall(regex, response, re.MULTILINE)

    if matches:
        # 콤마 제거 후 마지막 매칭값 반환
        ans = matches[-1].replace(",", "")
        return ans if ans else None

    # 만약 'Answer:' 태그가 없다면 전체 텍스트에서 가장 마지막 숫자를 시도
    additional_matches = re.findall(r"([0-9,.]+)", response)
    if additional_matches:
        ans = additional_matches[-1].replace(",", "")
        return ans if ans and ans != '.' else None

    return None

In [ ]:
### 수정해도 됩니다!
def run_benchmark_test(
        dataset,
        prompt: str,
        model: str = "llama-3.1-8b-instant",
        num_samples: int = 50,
        VERBOSE: bool = False
    ):
    correct = 0
    total   = 0
    results = []

    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["question"]
        correct_answer = float(re.findall(r'\d+(?:\.\d+)?', dataset[i]["answer"].split('####')[-1])[0])

        response = generate_response_using_Llama(
            prompt=prompt.format(question=question),
            model=model
        )

        if response:
            predicted_answer = extract_final_answer(response)

            # 빈 문자열이나 숫자가 아닌 경우 에러 방지
            if predicted_answer is not None:
                try:
                    # 콤마 제거 및 float 변환 시도
                    predicted_answer = float(str(predicted_answer).replace(",", ""))
                except ValueError:
                    predicted_answer = None
            else:
                predicted_answer = None

            diff = abs(predicted_answer - correct_answer) if predicted_answer is not None else 9999
            is_correct = diff < 1e-5

            if is_correct:
                correct += 1
            total += 1

            results.append({
                'question': question,
                'correct_answer': correct_answer,
                'predicted_answer': predicted_answer,
                'response': response,
                'correct': is_correct
            })

            if (i + 1) % 5 == 0:
                current_acc = correct/total if total > 0 else 0
                print(f"Progress: [{i+1}/{num_samples}]")
                print(f"Current Acc.: [{current_acc:.2%}]")

    return results, correct/total if total > 0 else 0

In [ ]:
def save_final_result(results: List[Dict[str, Any]], accuracy: float, filename: str) -> None:
    result_str = f"====== ACCURACY: {accuracy} ======\n\n"
    result_str += f"[Details]\n"

    for idx, result in enumerate(results):
        result_str += f"Question {idx+1}: {result['question']}\n"
        result_str += f"Correct Answer: {result['correct_answer']}\n"
        result_str += f"Predicted Answer: {result['predicted_answer']}\n"
        result_str += f"Correct: {result['correct']}\n\n"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(result_str)

#### Direct prompting with few-shot example

In [ ]:
def construct_direct_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )

    prompt = "Instruction:\nSolve the following mathematical question and generate ONLY the answer after a tag, 'Answer:' without any rationale.\n"

    for i in range(num_examples):
        cur_question = train_dataset['question'][i]
        cur_answer = train_dataset['answer'][i].split("####")[-1].strip()

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer:{cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [ ]:
### 어떤 방식으로 저장되는지 확인해보세요!
PROMPT = construct_direct_prompt(3)
VERBOSE = False

results, accuracy = run_benchmark_test(
    dataset=gsm8k_test,
    prompt=PROMPT,
    VERBOSE=VERBOSE,
    num_samples=10
)
save_final_result(results, accuracy, "example.txt")

 50%|█████     | 5/10 [00:01<00:01,  3.02it/s]

Progress: [5/10]
Current Acc.: [100.00%]


100%|██████████| 10/10 [00:10<00:00,  1.08s/it]

Progress: [10/10]
Current Acc.: [80.00%]


In [ ]:
# TODO: 0 shot, 3 shot, 5 shot direct prompting을 통해 벤치마크 테스트를 한 후, 각각 direct_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 direct_prompting_5.txt
# 항상 num_samples=50 입니다!
# 0-shot 실행
shot = 0
print(f"--- Running Direct Prompting - Shot: {shot} ---")
prompt_template = construct_direct_prompt(shot)
results, accuracy = run_benchmark_test(
    dataset=gsm8k_test,
    prompt=prompt_template,
    num_samples=50
)
# 결과 저장
save_final_result(results, accuracy, f"direct_prompting_{shot}.txt")
print(f"✅ Saved direct_prompting_{shot}.txt (Accuracy: {accuracy*100:.2f}%)")

--- Running Direct Prompting - Shot: 0 ---


 10%|█         | 5/50 [00:01<00:12,  3.59it/s]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:02<00:10,  3.78it/s]

Progress: [10/50]
Current Acc.: [70.00%]


...

Progress: [40/50]
Current Acc.: [80.00%]


 90%|█████████ | 45/50 [00:44<00:11,  2.30s/it]

Progress: [45/50]
Current Acc.: [77.78%]


100%|██████████| 50/50 [00:56<00:00,  1.14s/it]

Progress: [50/50]
Current Acc.: [78.00%]
✅ Saved direct_prompting_0.txt (Accuracy: 78.00%)


In [ ]:
# 3-shot 실행
shot = 3
print(f"--- Running Direct Prompting - Shot: {shot} ---")
prompt_template = construct_direct_prompt(shot)
results, accuracy = run_benchmark_test(
    dataset=gsm8k_test,
    prompt=prompt_template,
    num_samples=50
)
# 결과 저장
save_final_result(results, accuracy, f"direct_prompting_{shot}.txt")
print(f"✅ Saved direct_prompting_{shot}.txt (Accuracy: {accuracy*100:.2f}%)")

--- Running Direct Prompting - Shot: 3 ---


 10%|█         | 5/50 [00:17<02:49,  3.76s/it]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:36<02:28,  3.71s/it]

Progress: [10/50]
Current Acc.: [70.00%]


...

Progress: [40/50]
Current Acc.: [72.50%]


 90%|█████████ | 45/50 [02:38<00:18,  3.66s/it]

Progress: [45/50]
Current Acc.: [73.33%]


100%|██████████| 50/50 [02:56<00:00,  3.54s/it]

Progress: [50/50]
Current Acc.: [74.00%]
✅ Saved direct_prompting_3.txt (Accuracy: 74.00%)


In [ ]:
# 5-shot 실행
shot = 5
print(f"--- Running Direct Prompting - Shot: {shot} ---")
prompt_template = construct_direct_prompt(shot)
results, accuracy = run_benchmark_test(
    dataset=gsm8k_test,
    prompt=prompt_template,
    num_samples=50
)
# 결과 저장
save_final_result(results, accuracy, f"direct_prompting_{shot}.txt")
print(f"✅ Saved direct_prompting_{shot}.txt (Accuracy: {accuracy*100:.2f}%)")

--- Running Direct Prompting - Shot: 5 ---


 10%|█         | 5/50 [00:09<02:13,  2.98s/it]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:40<04:12,  6.30s/it]

Progress: [10/50]
Current Acc.: [60.00%]


...

Progress: [40/50]
Current Acc.: [75.00%]


 90%|█████████ | 45/50 [03:25<00:23,  4.69s/it]

Progress: [45/50]
Current Acc.: [71.11%]


100%|██████████| 50/50 [03:49<00:00,  4.59s/it]

Progress: [50/50]
Current Acc.: [72.00%]
✅ Saved direct_prompting_5.txt (Accuracy: 72.00%)


### Chain-of-Thought prompting with few-shot example
```text
[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
====================================================================================================
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
```

[Answer] 아래의 정답을 도출하는 과정을 예시로 달아주면 CoT의 few shot이 되겠죠?

In [ ]:
def construct_CoT_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )

    # 지시문에 '단계별로 생각하라'는 내용 추가.
    prompt = "Instruction:\nSolve the following mathematical questions by thinking step-by-step. Provide your reasoning followed by the final answer in the format 'Answer: [value]'.\n"

    for i, idx in enumerate(sampled_indices):
        cur_question = train_dataset['question'][idx]
        full_answer = train_dataset['answer'][idx]
        reasoning, final_ans = full_answer.split("####")

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer:\n{reasoning.strip()}\nTherefore, the answer is {final_ans.strip()}.\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [ ]:
# TODO: 0 shot, 3 shot, 5 shot CoT prompting을 통해 벤치마크 테스트를 한 후, 각각 CoT_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 CoT_prompting_5.txt
# 항상 num_samples=50 입니다!
# 0-shot 실행
shot = 0
print(f"--- Running CoT Prompting - Shot: {shot} ---")
prompt_template = construct_CoT_prompt(shot)
results, accuracy = run_benchmark_test(gsm8k_test, prompt_template, num_samples=50)
# 결과 저장
save_final_result(results, accuracy, f"CoT_prompting_{shot}.txt")
print(f"✅ Saved direct_prompting_{shot}.txt (Accuracy: {accuracy*100:.2f}%)")

--- Running CoT Prompting - Shot: 0 ---


 10%|█         | 5/50 [00:13<02:09,  2.88s/it]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:25<01:43,  2.58s/it]

Progress: [10/50]
Current Acc.: [60.00%]


...

Progress: [40/50]
Current Acc.: [72.50%]


 90%|█████████ | 45/50 [01:54<00:13,  2.64s/it]

Progress: [45/50]
Current Acc.: [75.56%]


100%|██████████| 50/50 [02:13<00:00,  2.67s/it]

Progress: [50/50]
Current Acc.: [76.00%]
✅ Saved direct_prompting_0.txt (Accuracy: 76.00%)


In [ ]:
# 3-shot 실행
shot = 3
print(f"--- Running CoT Prompting - Shot: {shot} ---")
prompt_template = construct_CoT_prompt(shot)
results, accuracy = run_benchmark_test(gsm8k_test, prompt_template, num_samples=50)
# 결과 저장
save_final_result(results, accuracy, f"CoT_prompting_{shot}.txt")
print(f"✅ Saved direct_prompting_{shot}.txt (Accuracy: {accuracy*100:.2f}%)")

--- Running CoT Prompting - Shot: 3 ---


 10%|█         | 5/50 [00:01<00:16,  2.80it/s]

Progress: [5/50]
Current Acc.: [100.00%]


 20%|██        | 10/50 [00:04<00:20,  1.96it/s]

Progress: [10/50]
Current Acc.: [80.00%]


...

Progress: [40/50]
Current Acc.: [65.00%]


 90%|█████████ | 45/50 [03:42<00:35,  7.02s/it]

Progress: [45/50]
Current Acc.: [66.67%]


100%|██████████| 50/50 [04:16<00:00,  5.14s/it]

Progress: [50/50]
Current Acc.: [70.00%]
✅ Saved direct_prompting_3.txt (Accuracy: 70.00%)


In [ ]:
# 5-shot 실행
shot = 5
print(f"--- Running CoT Prompting - Shot: {shot} ---")
prompt_template = construct_CoT_prompt(shot)
results, accuracy = run_benchmark_test(gsm8k_test, prompt_template, num_samples=50)
# 결과 저장
save_final_result(results, accuracy, f"CoT_prompting_{shot}.txt")
print(f"✅ Saved direct_prompting_{shot}.txt (Accuracy: {accuracy*100:.2f}%)")

--- Running CoT Prompting - Shot: 5 ---


 10%|█         | 5/50 [00:10<02:30,  3.34s/it]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [01:15<07:37, 11.44s/it]

Progress: [10/50]
Current Acc.: [70.00%]


...

Progress: [40/50]
Current Acc.: [72.50%]


 90%|█████████ | 45/50 [07:53<00:57, 11.50s/it]

Progress: [45/50]
Current Acc.: [71.11%]


100%|██████████| 50/50 [08:51<00:00, 10.62s/it]

Progress: [50/50]
Current Acc.: [72.00%]
✅ Saved direct_prompting_5.txt (Accuracy: 72.00%)


### Construct your prompt!!

목표: 본인만의 프롬프트를 통해 정답률을 더 끌어올려보기!
- gsm8k의 train 데이터셋에서 예시를 가져온 다음 (자유롭게!)
- 그 예시들에 대한 풀이 과정을 만들어주세요!
- 모든 것들이 자유입니다! Direct Prompting, CoT Prompting을 한 결과보다 정답률만 높으면 돼요.

In [ ]:
### 자유롭게 수정해도 됩니다! 완전히 새로 함수를 만들어도 돼요.
import random
import re

def construct_my_prompt(num_examples: int = 3):
    train_dataset = gsm8k_train

    # 1. 예시 샘플링 (너무 긴 것만 제외하고 다양하게 뽑기)
    candidates = []
    for i in range(len(train_dataset["question"])):
        ans = train_dataset["answer"][i]
        if "####" in ans:
            candidates.append(i)

    sampled_indices = random.sample(candidates, k=num_examples)

    # 2. 시스템 페르소나와 지시사항
    prompt = (
        "You are a Professor of Mathematics. Your goal is to solve complex math problems with perfect accuracy.\n"
        "Instructions:\n"
        "1. Read the question carefully.\n"
        "2. Break down the problem into small, logical steps.\n"
        "3. Explicitly write out your calculations.\n"
        "4. Double-check your arithmetic as you go.\n"
        "5. State the final answer clearly at the end.\n"
        "6. Format your final answer exactly as: Answer: <number>\n\n"
    )

    # 3. Few-shot 예시 구성
    for i, idx in enumerate(sampled_indices):
        q = train_dataset["question"][idx]
        full_answer = train_dataset["answer"][idx]
        reasoning, final_ans = full_answer.rsplit("####", 1)

        prompt += f"Problem {i+1}:\n{q}\n\n"
        prompt += "Solution:\n"
        prompt += f"Let's think step by step. {reasoning.strip()}\n"
        prompt += f"Answer: {final_ans.strip()}\n\n"
        prompt += "-" * 30 + "\n\n"

    # 4. 실제 문제 제시
    prompt += "Now, solve the following problem.\n"
    prompt += f"Problem:\n{{question}}\n\n"
    prompt += "Solution:\n"
    prompt += "Let's think step by step."

    return prompt

In [ ]:
# TODO: 만든 0 shot, 3 shot, 5 shot example과 프롬프트를 통해 벤치마크 테스트를 한 후, 각각 My_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 My_prompting_5.txt
# 항상 num_samples=50 입니다!

# 0-shot
shot = 0
print(f"--- Running My Prompting - Shot: {shot} ---")
prompt_template = construct_my_prompt(shot)
results, accuracy = run_benchmark_test(gsm8k_test, prompt_template, num_samples=50)
# 결과 저장
save_final_result(results, accuracy, f"My_prompting_{shot}.txt")
print(f"✅ Saved My_prompting_{shot}.txt (Accuracy: {accuracy*100:.2f}%)")

--- Running My Prompting - Shot: 0 ---


 10%|█         | 5/50 [00:02<00:18,  2.50it/s]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:05<00:23,  1.71it/s]

Progress: [10/50]
Current Acc.: [70.00%]


...

Progress: [40/50]
Current Acc.: [77.50%]


 90%|█████████ | 45/50 [01:54<00:15,  3.11s/it]

Progress: [45/50]
Current Acc.: [80.00%]


100%|██████████| 50/50 [02:09<00:00,  2.58s/it]

Progress: [50/50]
Current Acc.: [80.00%]
✅ Saved My_prompting_0.txt (Accuracy: 80.00%)


In [ ]:
# 3-shot
shot = 3
print(f"--- Running My Prompting - Shot: {shot} ---")
prompt_template = construct_my_prompt(shot)
results, accuracy = run_benchmark_test(gsm8k_test, prompt_template, num_samples=50)
# 결과 저장
save_final_result(results, accuracy, f"My_prompting_{shot}.txt")
print(f"✅ Saved My_prompting_{shot}.txt (Accuracy: {accuracy*100:.2f}%)")

--- Running My Prompting - Shot: 3 ---


 10%|█         | 5/50 [00:17<03:42,  4.95s/it]

Progress: [5/50]
Current Acc.: [100.00%]


 20%|██        | 10/50 [00:53<04:21,  6.53s/it]

Progress: [10/50]
Current Acc.: [90.00%]


...

Progress: [40/50]
Current Acc.: [90.00%]


 90%|█████████ | 45/50 [05:14<00:38,  7.63s/it]

Progress: [45/50]
Current Acc.: [91.11%]


100%|██████████| 50/50 [05:52<00:00,  7.05s/it]

Progress: [50/50]
Current Acc.: [90.00%]
✅ Saved My_prompting_3.txt (Accuracy: 90.00%)


In [ ]:
# 5-shot
shot = 5
print(f"--- Running My Prompting - Shot: {shot} ---")
prompt_template = construct_my_prompt(shot)
results, accuracy = run_benchmark_test(gsm8k_test, prompt_template, num_samples=50)
# 결과 저장
save_final_result(results, accuracy, f"My_prompting_{shot}.txt")
print(f"✅ Saved My_prompting_{shot}.txt (Accuracy: {accuracy*100:.2f}%)")

--- Running My Prompting - Shot: 5 ---


 10%|█         | 5/50 [00:49<06:42,  8.94s/it]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [01:54<08:15, 12.39s/it]

Progress: [10/50]
Current Acc.: [60.00%]


...

Progress: [40/50]
Current Acc.: [75.00%]


 90%|█████████ | 45/50 [08:58<01:01, 12.22s/it]

Progress: [45/50]
Current Acc.: [75.56%]


100%|██████████| 50/50 [10:03<00:00, 12.06s/it]

Progress: [50/50]
Current Acc.: [78.00%]
✅ Saved My_prompting_5.txt (Accuracy: 78.00%)


### 보고서 작성하기
#### 아래의 내용이 포함되면 됩니다!

1. Direct Prompting, CoT Prompting, My Prompting을 0 shot, 3 shot, 5 shot 정답률을 표로 보여주세요!
2. CoT Prompting이 Direct Prompting에 비해 왜 좋을 수 있는지에 대해서 서술해주세요!
3. 본인이 작성한 프롬프트 기법이 CoT에 비해서 왜 더 좋을 수 있는지에 대해서 설명해주세요!
4. 최종적으로, `PROMPTING.md`에 보고서를 작성해주세요!